# Business Entity Resolution — LOCAL Implementation Notebook (Windows)

**Runs on:** local Windows PC (Ultra 9 285H, 32 GB RAM, Intel Arc 140T iGPU) with OpenVINO for transformer inference.
**Sibling:** `entity_resolution.ipynb` is the SageMaker `ml.t3.medium` (2 vCPU / 4 GB) variant — untouched. This file is the local counterpart: bigger sample budgets, Windows-safe memory reporting, OpenVINO install + device check, and a full-scale blocking recipe sized for 32 GB instead of a t3.medium refusal stub.
**Plan ref:** `../ENTITY_RESOLUTION_PLAN.md` — Phase 0 → 0.5 → 1 → 2 → 3/4/5. All `.tsv` reads use `sep="\t"`.

Expected data layout (repo root on disk, already the case):
```
<DATA_ROOT>/train/train_source1.tsv
<DATA_ROOT>/train/train_source2.tsv
<DATA_ROOT>/train/train_source3.tsv
<DATA_ROOT>/train/train_ground_truth.tsv
<DATA_ROOT>/test/test_source1.tsv ...
```

## Which cells to run / skip (read this first)

**First run:** Run All, but expect exactly ONE red cell - the `RUN_FULL` assert (needs ~20 GB free + the full Phase-2 recipe implemented). A red `RUN_FULL` cell means the guardrail works.

| Cell | Verdict |
|---|---|
| `%pip install` | Run once - skip on re-runs unless the venv was recreated |
| Config (RAM budgets), scorer + unit test, val split, normalization lib | Must run every time |
| Skeleton run | Must run - the core pipeline |
| TSV write + validation | Must run - must print `PASS` |
| Drive detect | Must run for sharing - prints local-only when Drive is not mounted (set GDRIVE if letter differs) |
| Sync stage (push/pull) | Safe to always run - plain file copies via Drive for Desktop, resumes via manifest |
| France asserts | Optional - instant sanity check |
| FAISS ANN demo | Optional - smoke test at 100k-doc scale |
| OpenVINO device check | Must run (cheap) - confirms Arc `GPU` is visible |
| OV inference stub | Safe to always run - self-skips with a message until `models/minilm-ov` exists |
| `RUN_FULL` assert | **Do NOT run** - fails by design for now |
| Summary print | Optional |

In [ ]:
%pip install -q faiss-cpu rapidfuzz lightgbm openvino optimum-intel psutil

In [ ]:
import hashlib
import os
import platform
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

PROC = psutil.Process()


def rss_gb() -> float:
    return PROC.memory_info().rss / 1e9


TOTAL_GB = psutil.virtual_memory().total / 1e9
print(f"OS={platform.system()} CPUs={os.cpu_count()} RAM={TOTAL_GB:.1f} GB")

# ---- RAM-based budgets (32 GB local vs 4 GB t3.medium fallback) ----
if TOTAL_GB >= 16:
    SAMPLE_S1, POOL_DOCS = 20000, 100000  # ANN dense 4096-dim ~= 1.6 GB
    MODE = "local-32GB"
else:
    SAMPLE_S1, POOL_DOCS = 5000, 15000
    MODE = "low-ram-fallback"
TOP_K = 10
RANDOM_STATE = 42
RUN_FULL = False  # full 2.2M x 10M blocking: needs ~20 GB free, see scale-up cell
MODEL_DIR = Path("models/minilm-ov")  # OpenVINO-exported encoder (Phase 3)
OUT_DIR = Path("output-local"); OUT_DIR.mkdir(exist_ok=True)
# ---- Drive share via Drive for Desktop (sync stage only; pipeline never depends on it) ----
DRIVE_PARENT = "AmazonMLChallenge"  # your Drive folder (screenshot spelling)
DRIVE_SHARE = "shared"              # share subfolder (different name, as agreed)
MANIFEST = Path("sync_manifest.json")  # local-only sync state (gitignored)


def find_drive():
    cands = [os.environ.get("GDRIVE"), "G:/My Drive", "H:/My Drive",
             "I:/My Drive", "F:/My Drive", "E:/My Drive"]
    for c in cands:
        if c and Path(c).is_dir():
            return Path(c)
    return None
print(f"MODE={MODE} SAMPLE_S1={SAMPLE_S1} POOL_DOCS={POOL_DOCS}")

# ---- data root autodetect (repo checkout on disk) ----
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.home(),
                   Path.home() / "Amazon_ML_Challange"]
DATA_ROOT = None
for r in CANDIDATE_ROOTS:
    for c in [r / "dataset",
              r / "student_resource" / "student_resource" / "dataset",
              r / "student_resource" / "dataset"]:
        if (c / "train" / "train_source1.tsv").exists():
            DATA_ROOT = c
            break
    if DATA_ROOT is not None:
        break
print("DATA_ROOT =", DATA_ROOT)
assert DATA_ROOT is not None, "Dataset not found — run from the repo checkout."

In [ ]:
# ---- Drive-for-Desktop detect (None = offline; pipeline unaffected) ----
# No OAuth, no tokens: if Drive for Desktop is running (usually G:\My Drive),
# the share folder is just another local path. Set the GDRIVE env var if your
# drive letter differs. Re-run this cell after (re)connecting Drive.
GDRIVE = find_drive()
if GDRIVE is None:
    print("drive not mounted - local-only mode (outputs stay in output-local/).")
    SHARE_ROOT = None
else:
    SHARE_ROOT = GDRIVE / DRIVE_PARENT / DRIVE_SHARE
    print("drive path:", GDRIVE, "| share:", SHARE_ROOT)


## Phase 0 — Macro-F0.5 scorer (with unit test)
Singleton semantics: empty/empty → 1.0; any false merge on a singleton → 0.0.

In [ ]:
def f05_single(y_true: set, y_pred: set) -> float:
    tp = len(y_true & y_pred)
    prec = tp / len(y_pred) if y_pred else (1.0 if not y_true else 0.0)
    rec = tp / len(y_true) if y_true else (1.0 if not y_pred else 0.0)
    if prec + rec == 0:
        return 0.0
    return 1.25 * prec * rec / (0.25 * prec + rec)


def macro_f05(truth: dict, pred: dict) -> float:
    return sum(f05_single(set(truth[k]), set(pred.get(k, ()))) for k in truth) / len(truth)


# unit tests: PDF worked example S1-00001 -> P=2/3, R=1.0, F0.5=0.7142857
assert abs(f05_single({"S2-00047", "S3-00812"}, {"S2-00047", "S2-00193", "S3-00812"}) - 0.7142857) < 1e-6
assert f05_single(set(), set()) == 1.0
assert f05_single(set(), {"S2-1"}) == 0.0
assert f05_single({"S2-1"}, set()) == 0.0
print("scorer OK: pdf-example=0.7142857, singleton-empty=1.0, singleton-fp=0.0")

## Phase 0 — Validation split (hash-based, single chunked pass)
Deterministic 10% of S1 ids (`md5 % 10 == 0`), stratified *reporting* by match-count bucket. GT streams in chunks; only val rows kept (~220k).

In [ ]:
def in_val(s1: str) -> bool:
    return hashlib.md5(s1.encode()).digest()[0] % 10 == 0


def bucket(n: int) -> str:
    if n == 0:
        return "0-singleton"
    if n == 1:
        return "1"
    if n <= 3:
        return "2-3"
    if n <= 5:
        return "4-5"
    return "6+"


val_matches: dict = {}
total = 0
t0 = time.time()
for ch in pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", chunksize=200000):
    s1s = ch["source1_entity_id"].astype(str)
    ms = ch["matched_entity_ids"].fillna("").astype(str)
    for s1, m in zip(s1s, ms):
        total += 1
        if in_val(s1):
            m = m.strip()
            val_matches[s1] = [x for x in m.split(",") if x] if m else []
print(f"GT rows={total} val_S1={len(val_matches)} ({len(val_matches)/total:.1%}) in {time.time()-t0:.0f}s, rss={rss_gb():.2f} GB")
dist: dict = {}
for v in val_matches.values():
    b = bucket(len(v))
    dist[b] = dist.get(b, 0) + 1
print("val bucket distribution:", dist)

## Phase 0.5 — Walking skeleton (trivial PIN blocking, full pipeline)
Integration first: prove load → normalize → block → features → score → TSVs → validation PASS locally before building real blocking.

In [ ]:
US_IN_ABBR = {"corp": "corporation", "inc": "incorporated", "pvt": "private",
               "ltd": "limited", "rd": "road", "st": "street", "ave": "avenue",
               "blvd": "boulevard", "ste": "suite", "apt": "apartment",
               "mfg": "manufacturing", "ent": "enterprises", "co": "company"}
FR_ABBR = {"sarl": "societe responsabilite limitee", "sas": "societe actions simplifiee",
           "sa": "societe anonyme", "eurl": "entreprise unipersonnelle",
           "rue": "rue", "bd": "boulevard", "av": "avenue", "pl": "place",
           "imp": "impasse", "cedex": "cedex", "ste": "societe", "ets": "etablissements"}
ABBR = {**US_IN_ABBR, **FR_ABBR}  # open-set: FR included despite 0% train coverage
LEGAL_SUFFIX = set(ABBR) | {"corporation", "incorporated", "private", "limited",
                "company", "llc", "llp", "gmbh", "societe"}
PIN_RE = r"(?<!\d)(\d{5,6})(?!\d)"  # IN 6-digit, US/FR 5-digit


def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = s.lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    toks = [ABBR.get(t, t) for t in s.split()]
    return re.sub(r"\s+", " ", " ".join(toks)).strip()


def extract_pin(addr: str) -> str:
    m = re.search(PIN_RE, str(addr))
    return m.group(1) if m else ""


print("Lumay Bóral ->", normalize_text("Lumay Bóral"))
print("SARL Dupont, 12 Rue de la Paix, 75002 Paris ->", normalize_text("SARL Dupont, 12 Rue de la Paix, 75002 Paris"))

In [ ]:
from rapidfuzz import fuzz

sample_s1 = sorted(val_matches)[:SAMPLE_S1]
sample_set = set(sample_s1)
print(f"skeleton queries: {len(sample_s1)}, rss={rss_gb():.2f} GB")

# 1. load sample S1 rows (chunked scan, keep targets only)
s1_parts = []
for ch in pd.read_csv(DATA_ROOT / "train" / "train_source1.tsv", sep="\t", chunksize=200000):
    hit = ch[ch["entity_id"].astype(str).isin(sample_set)]
    if len(hit):
        s1_parts.append(hit)
s1_df = pd.concat(s1_parts, ignore_index=True)
assert len(s1_df) == len(sample_s1), (len(s1_df), len(sample_s1))
s1_df["norm_name"] = s1_df["business_name"].fillna("").map(normalize_text)
s1_df["norm_addr"] = s1_df["business_address"].fillna("").map(normalize_text)
s1_df["pin"] = s1_df["business_address"].fillna("").map(extract_pin)
print(f"sample S1 with PIN: {(s1_df['pin'] != '').sum()} / {len(s1_df)}, rss={rss_gb():.2f} GB")
pin_to_s1 = set(s1_df.loc[s1_df["pin"] != "", "pin"])

# 2. PIN-blocked pool scan over S2+S3 (vectorized extract per chunk; uses all cores)
t0 = time.time()
pool_parts = []
for name in ["train_source2.tsv", "train_source3.tsv"]:
    for ch in pd.read_csv(DATA_ROOT / "train" / name, sep="\t", chunksize=200000):
        pins = ch["business_address"].fillna("").str.extract(PIN_RE, expand=False)
        hit = ch[pins.isin(pin_to_s1)]
        if len(hit):
            pool_parts.append(hit)
pool_df = pd.concat(pool_parts, ignore_index=True).drop_duplicates("entity_id") if pool_parts else s1_df.iloc[0:0].copy()
pool_df["norm_name"] = pool_df["business_name"].fillna("").map(normalize_text)
pool_df["norm_addr"] = pool_df["business_address"].fillna("").map(normalize_text)
pool_df["pin"] = pool_df["business_address"].fillna("").map(extract_pin)
print(f"pool rows sharing a PIN: {len(pool_df)} in {time.time()-t0:.0f}s, rss={rss_gb():.2f} GB")

# 3. candidates per S1 (cap 40 = sanity bound)
pool_by_pin: dict = {}
for r in pool_df.itertuples():
    pool_by_pin.setdefault(r.pin, []).append(r.entity_id)
pool_idx = {r.entity_id: r for r in pool_df.itertuples()}
s1_idx = {r.entity_id: r for r in s1_df.itertuples()}
candidates = {s: list(pool_by_pin.get(s1_idx[s].pin, []))[:40] for s in sample_s1}
print(f"mean K={float(np.mean([len(v) for v in candidates.values()])):.1f}")

# 4. score (skeleton: untrained weighted sum) + threshold -> matches
def pair_score(a, b) -> float:
    ratio = fuzz.WRatio(a.norm_name, b.norm_name) / 100.0
    ta, tb = set(a.norm_name.split()), set(b.norm_name.split())
    jac = len(ta & tb) / max(1, len(ta | tb))
    pin = 1.0 if (a.pin and a.pin == b.pin) else 0.0
    return 0.5 * ratio + 0.3 * jac + 0.2 * pin


TAU_SKELETON = 0.5
matches = {}
for s in sample_s1:
    a = s1_idx[s]
    scored = sorted(((pool_idx[c], pair_score(a, pool_idx[c])) for c in candidates[s]),
                    key=lambda t: -t[1])
    matches[s] = [r.entity_id for r, sc in scored if sc >= TAU_SKELETON]
print(f"skeleton macro-F0.5 on sample: {macro_f05({s: val_matches[s] for s in sample_s1}, matches):.4f}")
print(f"rss={rss_gb():.2f} GB")

In [ ]:
# 5. write TSVs (tab-separated, utf-8) + validate
def write_id_list(path: Path, rows: dict, col: str):
    df = pd.DataFrame({"source1_entity_id": list(rows.keys()),
                       col: [",".join(rows[k]) for k in rows]})
    df.to_csv(path, sep="\t", index=False, encoding="utf-8")


write_id_list(OUT_DIR / "matching_results.tsv", matches, "matched_entity_ids")
write_id_list(OUT_DIR / "candidate_pairs.tsv", candidates, "candidate_entity_ids")
print("wrote", OUT_DIR / "matching_results.tsv", "and", OUT_DIR / "candidate_pairs.tsv")


def check_outputs(match_path: Path, cand_path: Path, required: set):
    issues = []
    m = pd.read_csv(match_path, sep="\t", keep_default_na=False)
    c = pd.read_csv(cand_path, sep="\t", keep_default_na=False)
    if list(m.columns) != ["source1_entity_id", "matched_entity_ids"]:
        issues.append(f"matching header: {list(m.columns)}")
    if list(c.columns) != ["source1_entity_id", "candidate_entity_ids"]:
        issues.append(f"candidate header: {list(c.columns)}")
    if set(m["source1_entity_id"]) != required or len(m) != len(required):
        issues.append("matching S1 row coverage mismatch")
    if set(c["source1_entity_id"]) != required or len(c) != len(required):
        issues.append("candidate S1 row coverage mismatch")
    valid = set(pool_df["entity_id"].astype(str))
    cmap = {}
    for _, r in c.iterrows():
        ids = [x for x in str(r["candidate_entity_ids"]).split(",") if x]
        if len(ids) != len(set(ids)):
            issues.append(f"dupes in candidates for {r['source1_entity_id']}")
        bad = [x for x in ids if not x.startswith(("S2-", "S3-")) or x not in valid]
        if bad:
            issues.append(f"bad candidate ids for {r['source1_entity_id']}: {bad[:3]}")
        cmap[r["source1_entity_id"]] = set(ids)
    for _, r in m.iterrows():
        ids = [x for x in str(r["matched_entity_ids"]).split(",") if x]
        if len(ids) != len(set(ids)):
            issues.append(f"dupes in matches for {r['source1_entity_id']}")
        if not set(ids) <= cmap.get(r["source1_entity_id"], set()):
            issues.append(f"match not in candidates for {r['source1_entity_id']}")
    return issues


issues = check_outputs(OUT_DIR / "matching_results.tsv",
                         OUT_DIR / "candidate_pairs.tsv", set(sample_s1))
print("VALIDATION:", "PASS" if not issues else f"FAIL {issues[:5]}")

In [ ]:
# ---- Sync stage: mirror to the Drive share folder (best-effort file copies) ----
# Local files are the source of truth. sync_manifest.json records hashes so only
# changed files move; reconnecting after offline days resumes the delta. Copies go
# via temp name + os.replace (atomic), then hash-verified. Any failure degrades to
# local-only - local outputs are always intact.
import hashlib
import json as _json
import os
import shutil


def md5_of(p: Path) -> str:
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(4 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_manifest() -> dict:
    try:
        return _json.loads(MANIFEST.read_text())
    except Exception:
        return {}


def push_file(local: Path, dest_dir: Path, key: str, manifest: dict, stats: dict):
    digest = md5_of(local)
    dest = dest_dir / local.name
    if manifest.get(key) == digest and dest.exists():
        stats["skip"] += 1
        return
    dest_dir.mkdir(parents=True, exist_ok=True)
    tmp = dest_dir / (local.name + ".part")
    shutil.copy2(local, tmp)
    os.replace(tmp, dest)  # atomic on the same filesystem
    assert md5_of(dest) == digest, f"verify failed: {local.name}"
    manifest[key] = digest
    stats["push"] += 1
    print(f"  pushed {local.name}")


def push_tree(local_dir: Path, dest_dir: Path, key_prefix: str, manifest: dict, stats: dict):
    if not local_dir.is_dir():
        return
    for p in sorted(local_dir.iterdir()):
        if p.is_file():
            push_file(p, dest_dir, f"{key_prefix}/{p.name}", manifest, stats)


def pull_tree(src_dir: Path, local_dir: Path, key_prefix: str, manifest: dict, stats: dict):
    if not src_dir.is_dir():
        return  # nothing shared yet - not an error
    local_dir.mkdir(parents=True, exist_ok=True)
    for p in sorted(src_dir.iterdir()):
        if not p.is_file() or p.suffix == ".part":
            continue
        key = f"{key_prefix}/{p.name}"
        digest = md5_of(p)
        dest = local_dir / p.name
        if manifest.get(key) == digest and dest.exists() and md5_of(dest) == digest:
            stats["skip"] += 1
            continue
        tmp = local_dir / (p.name + ".part")
        shutil.copy2(p, tmp)
        os.replace(tmp, dest)
        manifest[key] = digest
        stats["pull"] += 1
        print(f"  pulled {p.name}")


manifest = load_manifest()
if SHARE_ROOT is None:
    print("OFFLINE: local outputs kept; will sync once Drive is mounted.")
else:
    try:
        t0 = time.time()
        stats = {"push": 0, "skip": 0, "pull": 0}
        drive_base = GDRIVE / DRIVE_PARENT
        for p in sorted(OUT_DIR.glob("*.tsv")):
            push_file(p, SHARE_ROOT / "outputs", f"push:shared/outputs/{p.name}", manifest, stats)
        push_tree(Path("models/minilm-ov"), SHARE_ROOT / "models", "push:shared/models", manifest, stats)
        push_tree(Path("models/minilm-er"), SHARE_ROOT / "models", "push:shared/models", manifest, stats)
        lgbm = Path("models/lgbm.txt")
        if lgbm.exists():
            push_file(lgbm, SHARE_ROOT / "models", "push:shared/models/lgbm.txt", manifest, stats)
        pull_tree(drive_base / "models" / "minilm-ov", Path("models/minilm-ov"), "pull:models/minilm-ov", manifest, stats)
        pull_tree(drive_base / "models" / "minilm-er", Path("models/minilm-er"), "pull:models/minilm-er", manifest, stats)
        pull_tree(drive_base / "checkpoints" / "minilm-er", Path("models/minilm-er"), "pull:checkpoints/minilm-er", manifest, stats)
        MANIFEST.write_text(_json.dumps(manifest, indent=1))
        print(f"SYNC done in {time.time()-t0:.0f}s: {stats}")
    except OSError as e:
        print(f"SYNC interrupted ({e}) - local files intact; resumes next run (Drive may have disconnected).")


## Phase 1 — Normalization check incl. France

In [ ]:
assert normalize_text("SARL Dupont") == "societe responsabilite limitee dupont"
assert extract_pin("12 Rue de la Paix, 75002 Paris") == "75002"
assert extract_pin("Wadala, Mumbai 400031") == "400031"
assert normalize_text("Lumay Bóral") == normalize_text("Lumay Boral")
print("FR + accent normalization OK")

## Phase 2 — FAISS ANN demo at local scale (primary index, per plan)
Dense `IndexFlatIP` over hashed char 3–5g vectors. At 100k docs × 4096-dim ≈ 1.6 GB — comfortable on 32 GB, impossible on t3.medium. Recall denominator = matches present in pool.

In [ ]:
import faiss
from sklearn.feature_extraction.text import HashingVectorizer

try:
    faiss.omp_set_num_threads(os.cpu_count() or 8)
except Exception:
    pass
print("faiss threads:", faiss.omp_get_max_threads())

demo_pool = pool_df.head(POOL_DOCS).reset_index(drop=True)
pool_texts = (demo_pool["norm_name"] + " [SEP] " + demo_pool["norm_addr"]).tolist()
q_texts = [(s1_idx[s].norm_name + " [SEP] " + s1_idx[s].norm_addr) for s in sample_s1]
q_ids = [str(x) for x in demo_pool["entity_id"]]
pool_set = set(q_ids)

vec = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=4096,
                        alternate_sign=False, norm="l2")
t0 = time.time()
X = vec.transform(pool_texts).astype(np.float32).toarray()
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
Q = vec.transform(q_texts).astype(np.float32).toarray()
D, I = index.search(Q, TOP_K)
print(f"FAISS demo: pool={len(demo_pool)} dim={X.shape[1]} topK={TOP_K} in {time.time()-t0:.1f}s, rss={rss_gb():.2f} GB")

recalls, covered = [], 0
for i, s in enumerate(sample_s1):
    denom = [m for m in val_matches[s] if m in pool_set]
    if not denom:
        continue
    covered += 1
    hits = [q_ids[j] for j in I[i]]
    recalls.append(len(set(hits) & set(denom)) / len(denom))
print(f"pool coverage of sample GT: {covered}/{len(sample_s1)}")
if recalls:
    print(f"recall@{TOP_K} (in-pool denom) mean={np.mean(recalls):.3f}")
print("NOTE: skeleton pool is PIN-biased by construction — index smoke test, not a recall claim.")

## Phase 2 scale-up — full data on this 32 GB machine (not a stub refusal)
Unlike the SageMaker variant, full blocking is feasible here: per-country FAISS shard, chunked S1 queries (50k/chunk), streaming candidate writes. IVF-PQ keeps a 10M-doc index in RAM; FlatIP per shard works too if stair-cased. Flip `RUN_FULL=True` with ~20 GB free and implement per `ENTITY_RESOLUTION_PLAN.md` Phase 2, then enforce the recall gate (≥95% ceiling → minimize mean K per §1.6).

In [ ]:
assert RUN_FULL and psutil.virtual_memory().available / 1e9 > 20, (
    "Full-data blocking needs RUN_FULL=True and ~20 GB free RAM. "
    "Recipe: (1) per-country shard, (2) FAISS IVF-PQ/HNSW build, "
    "(3) 50k-chunk queries, (4) PIN/token backfill + rank-merge + adaptive cap, "
    "(5) recall/K gate on val.")
# full-scale code lands here (Phase 2 of ENTITY_RESOLUTION_PLAN.md)

## Phase 3 — OpenVINO transformer inference (Arc 140T iGPU)
OpenVINO accelerates *inference only*: fine-tune the bi-encoder once on a cloud GPU, export to OpenVINO IR, score candidates here on `GPU` (Arc 140T) with CPU fallback. FAISS/pandas/sklearn/LightGBM stay on CPU regardless.

In [ ]:
try:
    import openvino as ov
    core = ov.Core()
    print("openvino:", ov.__version__, "devices:", core.available_devices)
except Exception as e:
    core = None
    print("openvino unavailable:", type(e).__name__, e)

In [ ]:
# Export once (needs the fine-tuned encoder + GPU-box training first):
#   from optimum.intel import OVModelForFeatureExtraction
#   m = OVModelForFeatureExtraction.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", export=True)
#   m.save_pretrained(MODEL_DIR)  # openvino_model.xml + tokenizer

if core is None or not MODEL_DIR.exists():
    print("OV stub: no exported model yet — fine-tune on a GPU box, export to models/minilm-ov, re-run.")
else:
    from optimum.intel import OVModelForFeatureExtraction
    from transformers import AutoTokenizer
    device = "GPU" if "GPU" in core.available_devices else "CPU"
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    om = OVModelForFeatureExtraction.from_pretrained(MODEL_DIR, device=device)
    a = tok("Lumay Boral [SEP] 1056 Belden Avenue Akron OH", return_tensors="pt")
    b = tok("Lumay Boral Inc [SEP] 1056 Belden Ave Akon Ohio", return_tensors="pt")
    ea, eb = om(**a).last_hidden_state.mean(1).detach().numpy(), om(**b).last_hidden_state.mean(1).detach().numpy()
    cos = float(ea @ eb.T / (np.linalg.norm(ea) * np.linalg.norm(eb)))
    print(f"device={device} pair-cosine={cos:.3f} rss={rss_gb():.2f} GB")

## Phase 3 / 4 / 5 — Roadmap
- **Phase 3:** extend `pair_score` into pairwise features + LightGBM on val candidates (CPU, all cores); bi-encoder fine-tune on cloud GPU, inference here via the OpenVINO cell above.
- **Phase 4:** tune global `tau` on val macro-F0.5 (expect ~0.6–0.8); `max_score < tau` → empty list.
- **Phase 5:** chunked test inference → both TSVs → `validate_submission.py --check-ids` must PASS → submission zip.

In [ ]:
import json
print(json.dumps({"mode": MODE, "sample_S1": len(sample_s1), "pool_rows": len(pool_df),
                   "outputs": sorted(str(p) for p in OUT_DIR.glob("*.tsv")),
                   "rss_gb": round(rss_gb(), 2)}, indent=2))